In [6]:
import pandas as pd
from Bio import SeqIO

In [21]:
from ete3 import NCBITaxa

ncbi = NCBITaxa()

def get_family_taxid(taxid):
    """
    Return the family taxid and family name for a given NCBI taxid.

    Parameters
    ----------
    taxid : int

    Returns
    -------
    family
    """
    lineage = ncbi.get_lineage(taxid)
    ranks = ncbi.get_rank(lineage)
    print(lineage,ranks)

    family_taxid = next(
        (tid for tid in lineage if ranks.get(tid) == "family"),
        None
    )
    
    if family_taxid is None:
        return None, None

    family_name = ncbi.get_taxid_translator([family_taxid])[family_taxid]
    return family_name

In [54]:
a = "(((Psychodidae,(Ceratopogonidae,Culicidae)),((Sciaridae,Cecidomyiidae),(Stratiomyidae,(Syrphidae,((Tephritidae,Diopsidae),(Drosophilidae,(Glossinidae,(Muscidae,Calliphoridae)))))))));"

In [55]:
from ete3 import Tree
a=Tree(a)

In [9]:
df=df[df['Organism Name']!='Rhagoletis zephyria']
df=df[df['Organism Name']!='Bradysia coprophila']

In [94]:
!ls "$SCRATCH/extavour_lab/Everyone/Rishabh/ncbi_dataset/data/"GCF_963082655.1"

genomic.gff  protein.faa


In [27]:
for index, row in df.iterrows():
    df.loc[index,'genus']=row['Organism Name'].split(" ")[0]
## select only some members of genera Anopheles and Drosophila for analysis 
add=['Anopheles cruzii','Anopheles darlingi', 'Anopheles ziemanni','Anopheles gambiae','Anopheles funestus', 'Drosophila grimshawi',
    'Drosophila innubila','Drosophila virilis','Drosophila pseudoobscura','Drosophila arizonae','Drosophila willistoni','Drosophila ananassae','Drosophila kikkawai','Drosophila suzukii','Drosophila melanogaster','Drosophila grimshawi']


td=add+list(df[~df.genus.isin(["Anopheles","Drosophila"])]["Organism Name"])
td=df[df["Organism Name"].isin(td)]["Assembly Accession"]
len(td)

53

In [18]:
import os
for x in set(df['Assembly Accession']):
    if 'genomic.gff' not in os.listdir(f"/n/netscratch/extavour_lab/Everyone/Rishabh/ncbi_dataset/data/{x}/") or "protein.faa" not in os.listdir(f"/n/netscratch/extavour_lab/Everyone/Rishabh/ncbi_dataset/data/{x}"):
        !sh scripts/download_genomes.sh "$x"

In [47]:
from Bio import SeqIO
from collections import defaultdict
def longest_iso(ac):
    gff=pd.read_csv(f"/n/netscratch/extavour_lab/Everyone/Rishabh/ncbi_dataset/data/{ac}/genomic.gff",sep="\t",comment="#",header=None)
    def get_gene(x):
        return gff[gff[8].str.contains(x)].iloc[0,8].split(";gene=")[1].split(";")[0]
        
    input_file = open(f"/n/netscratch/extavour_lab/Everyone/Rishabh/ncbi_dataset/data/{ac}/protein.faa") 
    fasta = SeqIO.to_dict(SeqIO.parse(input_file, "fasta"))
    
    cds=gff[gff[2]=='CDS'].copy()
    cds['prot']=[x.split(";")[0].split("-")[1] for x in cds[8]]
    cds['gene']=[x.split(";gene=")[1].split(";")[0] for x in cds[8]]

    cds_gene={}
    gene_cds=defaultdict(set)
    for index, row in cds.iterrows():
        cds_gene[row.prot]=row.gene
        gene_cds[row.gene].add(row.prot)

    cds_len={x:len(str(fasta[x].seq)) for x in fasta}
    longest = []

    for gene, proteins in gene_cds.items():
        max_len=max([cds_len[x] for x in proteins])
        longest.append([x for x in proteins if cds_len[x]==max_len][0])
    longest_records = []

    for x in longest:
        record = fasta[x]
        record.id = f"{ac};{x}"
        record.name = f"{ac};{x}"
        record.description = f"{ac};{x}"
        longest_records.append(record)

    
    SeqIO.write(
        longest_records,
        f"/n/netscratch/extavour_lab/Everyone/Rishabh/ncbi_dataset/data/{ac}/longest_isoform.fasta",
        "fasta"
    )

In [47]:
!mkdir "$SCRATCH/extavour_lab/Everyone/Rishabh/diamond_clustered_dipteran_results"

In [43]:
import os
for x in os.listdir("/n/netscratch/extavour_lab/Everyone/Rishabh/diamond_clustered_dipteran_results/"):
    if 'GCF_' not in x:
        a=f"/n/netscratch/extavour_lab/Everyone/Rishabh/diamond_clustered_dipteran_results/{x}"
        !rm "$a"

In [52]:
for x in td:
    print(x)
    !sbatch scripts/diamond_clustered_all_dipt.sh "$x"

GCF_024334085.1
Submitted batch job 22443446
GCF_000001215.4
Submitted batch job 22443447
GCF_002204515.2
Submitted batch job 22443448
GCF_943734735.2
Submitted batch job 22443449
GCF_035046485.1
Submitted batch job 22443450
GCF_023373825.1
Submitted batch job 22443452
GCF_905115235.1
Submitted batch job 22443454
GCF_015732765.1
Submitted batch job 22443455
GCF_016801865.2
Submitted batch job 22443467
GCF_009870125.1
Submitted batch job 22443468
GCF_017639315.1
Submitted batch job 22443469
GCF_043229965.1
Submitted batch job 22443470
GCF_028554725.1
Submitted batch job 22443471
GCF_028408465.1
Submitted batch job 22443472
GCF_030788295.1
Submitted batch job 22443473
GCF_943734845.2
Submitted batch job 22443474
GCF_943734745.1
Submitted batch job 22443476
GCF_042242935.1
Submitted batch job 22443478
GCF_945859705.1
Submitted batch job 22443479
GCF_016617805.1
Submitted batch job 22443480
GCF_050003625.1
Submitted batch job 22443481
GCF_958450345.1
Submitted batch job 22443483
GCF_018902

In [36]:
!sbatch scripts/diamond_clustered_all_dipt.sh "$x"

Submitted batch job 22436658


In [48]:
for ac in set(df['Assembly Accession']):
    print(ac)
    longest_iso(ac)

GCF_029784165.1
GCF_943734745.1
GCF_040869045.1
GCF_024586455.1
GCF_024139115.2
GCF_030788295.1
GCF_016617805.1
GCF_009176525.2
GCF_024334085.1
GCF_943734685.1
GCF_023558535.2
GCF_014805625.2
GCF_943734655.1
GCF_043229965.1
GCF_028408465.1
GCF_030247185.1
GCF_016746365.2
GCF_015732765.1
GCF_016920715.1
GCF_018902025.1
GCF_036172545.1
GCF_013731165.1
GCF_030504385.1
GCF_030247195.1
GCF_943734735.2
GCF_905115235.1
GCF_014743375.2
GCF_943737925.1
GCF_016746245.2
GCF_965234385.1
GCF_943734845.2
GCF_943735745.2
GCF_943734755.1
GCF_945859705.1
GCF_029784155.1
GCF_013141755.1
GCF_943734635.1
GCF_009650485.2
GCF_014529535.2
GCF_000347755.3
GCF_028554725.1
GCF_001654015.2
GCF_008121235.1
GCF_016801865.2
GCF_004382145.1
GCF_958450345.1
GCF_009870125.1
GCF_943734705.1
GCF_029784135.1
GCF_030179895.1
GCF_022045245.1
GCF_030179915.1
GCF_017562075.2
GCF_943734695.1
GCF_000001215.4
GCF_947086385.1
GCF_030179905.1
GCF_004382195.2
GCF_945859685.1
GCF_023373825.1
GCF_002237135.1
GCF_035046485.1
GCF_0235

In [5]:
!rm -r "$SCRATCH/extavour_lab/Everyone/Rishabh/ncbi_dataset/all_dipteran_proteins_one_per_iso"

In [4]:
!scancel -n DIAMOND

In [11]:
!ls "$SCRATCH/extavour_lab/Everyone/Rishabh/ncbi_dataset/all_dipteran_proteins_one_per_iso"


10.fasta  13.fasta  16.fasta  19.fasta	2.fasta  5.fasta  8.fasta
11.fasta  14.fasta  17.fasta  1.fasta	3.fasta  6.fasta  9.fasta
12.fasta  15.fasta  18.fasta  20.fasta	4.fasta  7.fasta


In [6]:
%%bash
indir="$SCRATCH/extavour_lab/Everyone/Rishabh/ncbi_dataset/data"
outdir="$SCRATCH/extavour_lab/Everyone/Rishabh/ncbi_dataset/all_dipteran_proteins_one_per_iso"

mkdir -p "$outdir"

# Collect sorted input FASTA paths
mapfile -d '' files < <(
    find "$indir" -type f -name "longest_isoform.fasta" -print0 | sort -z
)

n=${#files[@]}
bins=20

echo "Found $n input FASTA files"

# Remove old outputs
for b in $(seq 1 "$bins"); do
    : > "$outdir/${b}.fasta"
done

# Split files as evenly as possible across 5 bins
for i in "${!files[@]}"; do
    bin=$(( i * bins / n + 1 ))
    cat "${files[$i]}" >> "$outdir/${bin}.fasta"
done

echo "Wrote concatenated FASTA bins to:"
for b in $(seq 1 "$bins"); do
    outfile="$outdir/${b}.fasta"
    echo "$outfile"
    echo -n "Number of proteins in ${b}.fasta: "
    grep -c "^>" "$outfile"
done

Found 86 input FASTA files
Wrote concatenated FASTA bins to:
/n/netscratch/extavour_lab/Everyone/Rishabh/ncbi_dataset/all_dipteran_proteins_one_per_iso/1.fasta
Number of proteins in 1.fasta: 64866
/n/netscratch/extavour_lab/Everyone/Rishabh/ncbi_dataset/all_dipteran_proteins_one_per_iso/2.fasta
Number of proteins in 2.fasta: 72980
/n/netscratch/extavour_lab/Everyone/Rishabh/ncbi_dataset/all_dipteran_proteins_one_per_iso/3.fasta
Number of proteins in 3.fasta: 56352
/n/netscratch/extavour_lab/Everyone/Rishabh/ncbi_dataset/all_dipteran_proteins_one_per_iso/4.fasta
Number of proteins in 4.fasta: 68638
/n/netscratch/extavour_lab/Everyone/Rishabh/ncbi_dataset/all_dipteran_proteins_one_per_iso/5.fasta
Number of proteins in 5.fasta: 64959
/n/netscratch/extavour_lab/Everyone/Rishabh/ncbi_dataset/all_dipteran_proteins_one_per_iso/6.fasta
Number of proteins in 6.fasta: 55968
/n/netscratch/extavour_lab/Everyone/Rishabh/ncbi_dataset/all_dipteran_proteins_one_per_iso/7.fasta
Number of proteins in 7.

In [9]:
!mkdir "$SCRATCH/extavour_lab/Everyone/Rishabh/diamond_clustered_dipteran_results"

In [15]:
df=pd.read_csv("outputs/manual_inspection_table.tsv",sep="\t",index_col=0)
td=[0, 1, 2, 3, 4, 14, 17, 18, 22, 24, 32, 34, 35, 38, 40, 42, 50, 62, 64, 66, 70, 76, 78, 80, 81, 82, 93, 97, 106, 111, 115, 116, 120, 128, 132, 133, 134, 137, 140, 144, 148, 150, 151, 153, 161, 165, 173, 178, 185, 191, 193, 194, 197, 203, 209, 211, 218, 229, 231, 233, 237, 240, 241, 243, 250, 254, 255, 268, 284, 286, 295, 296, 297, 302, 306, 310, 317, 319, 332, 341, 342, 351, 354, 364, 372, 373, 381, 398, 405, 417, 418, 423, 425, 435, 469, 470, 483, 493, 509, 513, 537, 547, 551, 583, 587, 606, 616, 627, 632, 633, 664, 666, 694, 721, 725, 748, 757, 768, 781, 805, 828, 831, 832, 845, 858, 876]
df['redo']=[int(str(x).split(".")[0]) in td for x in df.index]

In [19]:
df.to_csv("outputs/manual_inspection_table.tsv",sep="\t")

In [5]:
!scancel -n diamond

In [7]:
!seff 21030589	

Job ID: 21030589
Cluster: odyssey
User/Group: rkapoor/extavour_lab
State: CANCELLED (exit code 0)
Nodes: 1
Cores per node: 110
CPU Utilized: 22-23:29:06
CPU Efficiency: 94.94% of 24-04:53:30 core-walltime
Job Wall-clock time: 05:16:51
Memory Utilized: 33.75 GB
Memory Efficiency: 8.44% of 400.00 GB (400.00 GB/node)


In [8]:
for i in range(1,21):
    print(i)
    !sbatch scripts/diamond_clustered_all_dipt.sh "$i"
    

1
Submitted batch job 21152315
2
Submitted batch job 21152318
3
Submitted batch job 21152327
4
Submitted batch job 21152333
5
Submitted batch job 21152348
6
Submitted batch job 21152354
7
Submitted batch job 21152359
8
Submitted batch job 21152393
9
Submitted batch job 21152398
10
Submitted batch job 21152402
11
Submitted batch job 21152404
12
Submitted batch job 21152408
13
Submitted batch job 21152414
14
Submitted batch job 21152433
15
Submitted batch job 21152438
16
Submitted batch job 21152442
17
Submitted batch job 21152448
18
Submitted batch job 21152452
19
Submitted batch job 21152453
20
Submitted batch job 21152454


In [2]:
!scancel -n diamond

In [96]:
!awk 'BEGIN{FS=OFS="\t"} {$1=substr($1,1,index($1,"_")-1)}1' "ncbi_dataset/data/GCF_014529535.2/Galaxy15-[BRAKER3 on dataset 2 and 14_ GTF Annotation].gtf" > "ncbi_dataset/data/GCF_014529535.2/braker.gtf"